[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giulia-open-lab/sionna-rt-course/blob/main/notebooks/02_lab_student.ipynb)

# Lab: ray tracing for radio propagation with Sionna RT

**Hands-on lab** for the MSc course *Radio Propagation*: about 2 hours, alone or in pairs, after
the lecture demo (`01_instructor_demo.ipynb`).

## Learning objectives

After this lab you can:

1. Implement the Friis formula and obtain path gains from the path coefficients computed by
   Sionna RT (agreement within 0.1 dB).
2. Implement the Fresnel reflection coefficient for the correct polarization and the two-ray
   model; find the breakpoint distance and the path-loss exponents before and after it.
3. Run `PathSolver` in an urban scene, compute the RMS delay spread and the coherence bandwidth,
   and find the dominant propagation mechanism by switching mechanisms off.
4. Run `RadioMapSolver`, compute the percentage of outdoor area above a received-power
   threshold, and explain the effect of the number of interactions, diffraction and refraction.
5. (Bonus) Assess the effect of the frequency or of the building materials on coverage.

## Prerequisites

- The lecture demo of this course (`01_instructor_demo.ipynb`).
- **Propagation:** Friis equation, dB and dBm; Fresnel reflection coefficients (TE/TM) and the
  Brewster angle; two-ray model and breakpoint; power delay profile, RMS delay spread, coherence
  bandwidth; path-loss exponent.
- **Python:** functions, NumPy arrays (`np.log10`, `np.sqrt`, boolean masks), `np.polyfit`.

## How this lab works

| Exercise | Topic | Time |
|---|---|---|
| 0 | Setup and environment check | 5 min |
| 1 | Free space: Friis | 15 min |
| 2 | Two-ray model over a real ground | 30 min |
| 3 | Multipath in a street canyon | 35 min |
| 4 | Radio map and coverage | 30 min |
| 5 | Bonus: frequency or materials | extra |

- **Run the setup cells first** (setup, helpers, checks). After that every exercise
  builds its own scene, so you can skip an exercise if you are stuck.
- Parts to complete are marked **`TODO [Ex N.M]`**: replace the line
  `raise NotImplementedError(...)` with your code. Until you do, the cells that use that part stop
  with a `NotImplementedError` that names the exercise: this is expected.
- After each part, a **check cell** prints *Correct*, *Not correct* with a hint, or
  *Exercise N.M not completed yet*. The checks use tolerances and never show the solution.
- **Before running: what do you expect?** Some cells ask for a prediction: write it *before*
  running the code below it.
- Answer the questions in the **Your answer** cells, using numbers from your own results and a
  physical explanation.
- In Exercises 1 and 2 the Sionna calls are given and you write the physics. In Exercises 3 and 4
  you also write the Sionna calls, guided by hints and by the
  [Sionna RT API documentation](https://nvlabs.github.io/sionna/rt/api/rt.html).
- **Conventions:** path gain $G=\lvert a\rvert^2$ (linear, $\le 1$), $G_\mathrm{dB}=10\log_{10}G$;
  received power $P_\mathrm{rx}\,[\mathrm{dBm}]=P_\mathrm{tx}\,[\mathrm{dBm}]+G_\mathrm{dB}$
  (isotropic antennas, 0 dBi). SI units in the code (m, Hz, s, W).
- At the end, hand in the notebook (see *How to submit*).

In [ ]:
# Setup: install Sionna RT if it is missing (Colab), select the Mitsuba variant, print versions
import importlib.util
import os
import sys

SIONNA_RT_VERSION = "2.2.0"   # version this notebook was written and tested with
FORCE_CPU = False             # True: use the CPU (LLVM) variant even if a GPU is available

if importlib.util.find_spec("sionna") is None:
    if "google.colab" in sys.modules:
        print(f"Installing sionna-rt=={SIONNA_RT_VERSION}. The runtime will restart: "
              "when it has restarted, run this cell again.")
        os.system(f"pip install -q sionna-rt=={SIONNA_RT_VERSION}")
        os.kill(os.getpid(), 5)   # restart the runtime, as in the official Sionna tutorials
    else:
        raise ImportError("sionna-rt is not installed: run 'pip install -r requirements.txt' "
                          "(see the README)")

import mitsuba as mi
if FORCE_CPU:
    mi.set_variant("llvm_ad_mono_polarized")   # must be set before importing sionna.rt
try:
    import sionna.rt   # selects the CUDA variant if a GPU is available, otherwise LLVM (CPU)
except ImportError as err:
    if "LLVM" in str(err):
        raise RuntimeError("No GPU was found and the LLVM library needed by the CPU variant is "
                           "missing. Install it (on Ubuntu, e.g. 'sudo apt install libllvm18') or "
                           "ask the lab administrators; see the README.") from err
    raise

import drjit as dr
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import epsilon_0, speed_of_light
from sionna.rt import (Camera, ITURadioMaterial, InteractionType, PathSolver, PlanarArray,
                       RadioMapSolver, Receiver, SceneObject, Transmitter, load_scene,
                       subcarrier_frequencies)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3})
SEED = 42   # seed passed to every ray-tracing call

device = "GPU (CUDA)" if mi.variant().startswith("cuda") else "CPU (LLVM)"
print(f"Python {sys.version.split()[0]} | sionna-rt {sionna.rt.__version__} | "
      f"Mitsuba {mi.__version__} | Dr.Jit {dr.__version__}")
print(f"NumPy {np.__version__} | matplotlib {matplotlib.__version__}")
print(f"Mitsuba variant: {mi.variant()}  ->  running on {device}")
if sionna.rt.__version__ != SIONNA_RT_VERSION:
    print(f"WARNING: this notebook was tested with sionna-rt {SIONNA_RT_VERSION}")

### Helpers and automatic checks

Run the next two cells. The first defines a few helpers used by the exercises (you do not need to
modify or fully read them). The second defines the automatic checks: **do not modify it**.

In [ ]:
# ---------------------------------------------------------------------- helper functions (given)
def flat_ground(x_min, x_max, y_min, y_max, tile=100.0):
    # Flat ground at z = 0 made of square tiles (two triangles each), as a Mitsuba mesh.
    # (One huge triangle would lose some reflections because of single-precision errors.)
    xs = np.arange(x_min, x_max + tile/2, tile)
    ys = np.arange(y_min, y_max + tile/2, tile)
    X, Y = np.meshgrid(xs, ys, indexing="ij")
    vertices = np.stack([X.ravel(), Y.ravel(), np.zeros(X.size)], axis=1)
    ny = len(ys)
    faces = []
    for i in range(len(xs) - 1):
        for j in range(ny - 1):
            v0, v1, v2, v3 = i*ny + j, (i+1)*ny + j, (i+1)*ny + j + 1, i*ny + j + 1
            faces += [[v0, v1, v2], [v0, v2, v3]]
    faces = np.array(faces)
    mesh = mi.Mesh("ground", vertex_count=len(vertices), face_count=len(faces),
                   has_vertex_normals=False, has_vertex_texcoords=False)
    params = mi.traverse(mesh)
    params["vertex_positions"] = mi.Float(vertices.ravel().astype(np.float32))
    params["faces"] = mi.UInt32(faces.ravel().astype(np.uint32))
    params.update()
    return mesh


TYPE_LETTER = {int(InteractionType.SPECULAR): "R", int(InteractionType.DIFFRACTION): "D",
               int(InteractionType.REFRACTION): "T", int(InteractionType.DIFFUSE): "S"}

def paths_of_rx(paths, rx):
    # Valid paths of receiver index `rx` (one transmitter, single antennas): complex baseband
    # coefficients a_i, delays tau_i [s], and interaction strings ("LoS", "R", "R-D", ...)
    a, tau = paths.cir(normalize_delays=False, out_type="numpy")
    a, tau = a[rx, 0, 0, 0, :, 0], tau[rx, 0, :]
    valid = paths.valid.numpy()[rx, 0, :]
    inter = paths.interactions.numpy()[:, rx, 0, :]
    types = ["-".join(TYPE_LETTER[int(t)] for t in inter[:, i] if t != 0) or "LoS"
             for i in range(inter.shape[1])]
    keep = np.flatnonzero(valid)
    return a[keep], tau[keep], [types[i] for i in keep]


# Street-canyon scene of Exercises 3-5
TX_CANYON = [-50.0, 0.0, 10.0]       # small cell on a 10 m pole in the main street
RX_LOS = [20.0, 0.0, 1.5]            # receiver in the main street (line of sight)
RX_NLOS = [-23.0, -25.0, 1.5]        # receiver in a cross street, behind a building corner
P_TX_DBM = 30.0                      # transmit power [dBm]

def make_canyon_scene(frequency=3.5e9):
    # simple_street_canyon with the transmitter and the two receivers, isotropic V antennas
    scene = load_scene(sionna.rt.scene.simple_street_canyon)
    scene.frequency = frequency
    scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
    scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
    scene.add(Transmitter(name="tx", position=TX_CANYON, power_dbm=P_TX_DBM, display_radius=3.0))
    scene.add(Receiver(name="rx-los", position=RX_LOS, display_radius=3.0))
    scene.add(Receiver(name="rx-nlos", position=RX_NLOS, display_radius=3.0))
    return scene

cam_canyon = Camera(position=[-5.0, -60.0, 290.0], look_at=[-5.0, -5.0, 0.0])


def outdoor_mask(scene, rm, z=1.5):
    # True for the radio-map cells with no building above them (vertical ray cast from above)
    centers = rm.cell_centers.numpy().reshape(-1, 3)
    z_top = float(scene.mi_scene.bbox().max.z) + 10.0
    origins = mi.Point3f(centers[:, 0], centers[:, 1], np.full(len(centers), z_top, dtype=np.float32))
    hit = scene.mi_scene.ray_intersect(mi.Ray3f(origins, mi.Vector3f(0.0, 0.0, -1.0)))
    return (hit.t.numpy() >= z_top - z).reshape(rm.cell_centers.shape[:2])


def show_map(ax, values_db, rm, outdoor, title, vmin=-130, vmax=-30, label="dBm"):
    # Top view of a radio-map quantity in dB/dBm; buildings in grey, cells without signal in white
    centers = rm.cell_centers.numpy()
    extent = [centers[0, 0, 0] - 0.5, centers[0, -1, 0] + 0.5, centers[0, 0, 1] - 0.5, centers[-1, 0, 1] + 0.5]
    shown = np.where(np.isfinite(values_db), values_db, np.nan)
    im = ax.imshow(shown, origin="lower", extent=extent, vmin=vmin, vmax=vmax, cmap="viridis")
    ax.imshow(np.where(outdoor, np.nan, 1.0), origin="lower", extent=extent, cmap="Greys",
              vmin=0, vmax=1.6)
    ax.plot(TX_CANYON[0], TX_CANYON[1], "r^", ms=7)
    ax.plot([RX_LOS[0], RX_NLOS[0]], [RX_LOS[1], RX_NLOS[1]], "mx", ms=7, mew=2)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("x [m]")
    ax.grid(False)
    plt.colorbar(im, ax=ax, shrink=0.8, label=label)

In [ ]:
# ---------------------------------------------- automatic checks: do NOT modify this cell
# They compare your results with reference values computed in advance (with tolerances).
# The reference values are encoded so that they are not read by accident.
import base64
import json
_REF = json.loads(base64.b64decode("eyJmcmlpc18zNSI6IFstNDMuMzI5MTQsIC04My4zMjkxNCwgLTEwMy4zMjkxNF0sICJmcmlpc18yOCI6IC0xMDEuMzkwOTQsICJyX3YiOiBbWzAuNTY1MDg3ODIsIC0wLjAxNzcxNzI4XSwgWzAuNDQ5MDQ0MDksIC0wLjAyMDAxMzc5XSwgWy0wLjIwNjY2NjU2LCAtMC4wMjMwMDkxNV0sIFstMC44NzU5ODU1NCwgLTAuMDA1NTc4MzhdXSwgInJfaCI6IFtbLTAuNTc0NjY5MzgsIDAuMDE3NTEyNDhdLCBbLTAuNjcwMjczODEsIDAuMDE0OTI5NTddLCBbLTAuOTA1OTEzNzUsIDAuMDA1MDQ3NTddLCBbLTAuOTkwMTE1NjYsIDAuMDAwNTU1MTJdXSwgImRfYiI6IFsyMTAuMTQ1NCwgMjY2Ljg1MTNdLCAibl9yYW5nZSI6IFtbMS44LCAyLjRdLCBbMy41LCA0LjJdXSwgImdfMzEiOiBbLTc0LjczLCAtOTAuODcsIC05MS4xN10sICJyZWZyYWN0aW9uX2hpbnQiOiAtODYuMCwgImdfMzQiOiB7ImFsbCI6IC05MC44NywgIm5vIGRpZmZyYWN0aW9uIjogLTkxLjE3LCAibm8gcmVmbGVjdGlvbiI6IC0xMDguNTl9LCAicmVhY2hlZF80MSI6IDI5LjAsICJjb3ZlcmFnZV80NCI6IDk4LjI1LCAiY292ZXJhZ2VfNWEiOiA1My43LCAiY292ZXJhZ2VfNWIiOiA3Ny4zfQ==").decode())

def _attempt(label, func, *args, **kwargs):
    # Call a student function; returns (True, result) or (False, None) after printing why
    try:
        return True, func(*args, **kwargs)
    except NotImplementedError:
        print(f"Exercise {label} not completed yet.")
    except Exception as err:
        print(f"Not correct (Ex {label}): your code raised {type(err).__name__}: {err}")
    return False, None

def _verdict(label, ok, hint=""):
    print(f"Correct (Ex {label})." if ok else f"Not correct (Ex {label}). Hint: {hint}")

def _close(x, ref, tol):
    x = np.asarray(x, dtype=complex if np.iscomplexobj(ref) else float)
    return x.shape == np.shape(ref) and np.allclose(x, ref, atol=tol, rtol=0)


def check_ex0():
    ok_version = sionna.rt.__version__ == SIONNA_RT_VERSION
    ok_variant = mi.variant() in ("cuda_ad_mono_polarized", "llvm_ad_mono_polarized")
    scene = load_scene()
    scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
    scene.rx_array = scene.tx_array
    scene.add(Transmitter(name="tx", position=[0.0, 0.0, 10.0]))
    scene.add(Receiver(name="rx", position=[50.0, 0.0, 10.0]))
    ok_paths = bool(PathSolver()(scene, max_depth=0).valid.numpy().any())
    print(f"sionna-rt {sionna.rt.__version__}, variant {mi.variant()}")
    _verdict("0", ok_version and ok_variant and ok_paths,
             "the environment is not as expected: re-run the setup cell and read its messages.")


def check_ex1_1(friis_path_gain_db):
    ok, g = _attempt("1.1", friis_path_gain_db, np.array([1.0, 100.0, 1000.0]), 3.5e9)
    if not ok:
        return
    ref = np.array(_REF["friis_35"])
    ok2, g28 = _attempt("1.1", friis_path_gain_db, 100.0, 28e9)
    if not ok2:
        return
    good = _close(g, ref, 0.01) and abs(float(np.squeeze(g28)) - _REF["friis_28"]) < 0.01
    hint = "G_dB = 20 log10(lambda / (4 pi d)), with lambda = c / f in metres and f in Hz."
    if np.shape(g) == ref.shape and np.allclose(g, -ref, atol=0.01):
        hint = "you returned a positive path loss: this lab uses the path gain, which is negative in dB."
    elif np.shape(g) == ref.shape and np.allclose(g, ref/2, atol=0.01):
        hint = "lambda/(4 pi d) is an amplitude ratio: use 20 log10, not 10 log10."
    elif np.shape(g) != ref.shape:
        hint = "your function must accept a NumPy array of distances and return an array (use np.log10)."
    _verdict("1.1", good, hint)


def check_ex1_2(path_gain_db):
    ok, g1 = _attempt("1.2", path_gain_db, np.array([1e-4 + 0j]))
    if not ok:
        return
    ok, g3 = _attempt("1.2", path_gain_db, np.array([1e-4, 1e-4j, -1e-4]))
    if not ok:
        return
    good = abs(float(g1) - (-80.0)) < 1e-3 and abs(float(g3) - (-75.22879)) < 1e-3
    hint = "G = sum of the path powers |a_i|^2 (linear), then 10 log10."
    if abs(float(g3) - (-80.0)) < 1e-3:
        hint = "add the powers |a_i|^2 of the paths, not their complex amplitudes."
    elif abs(float(g3) - (-70.45757)) < 1e-3:
        hint = "|a_i| is an amplitude: add the powers |a_i|^2."
    _verdict("1.2", good, hint)


def check_ex1_3(friis_path_gain_db, path_gain_db, a_paths, distances, frequency):
    ok, g_rt = _attempt("1.2", lambda: np.array([path_gain_db(a) for a in a_paths]))
    if not ok:
        return
    ok, g_fs = _attempt("1.1", friis_path_gain_db, distances, frequency)
    if not ok:
        return
    diff = np.max(np.abs(np.asarray(g_rt) - np.asarray(g_fs)))
    _verdict("1.3", diff < 0.1, f"the largest difference is {diff:.2f} dB: check Ex 1.1 and 1.2 first.")


def check_ex2_1(reflection_coefficient):
    eta = 13.23 - 1.385j
    theta = np.radians([10.0, 45.0, 80.0, 89.0])
    ref_v = np.array([complex(*z) for z in _REF["r_v"]])
    ref_h = np.array([complex(*z) for z in _REF["r_h"]])
    ok, rv = _attempt("2.1", reflection_coefficient, theta, eta, "V")
    if not ok:
        return
    ok, rh = _attempt("2.1", reflection_coefficient, theta, eta, "H")
    if not ok:
        return
    good = _close(rv, ref_v, 1e-6) and _close(rh, ref_h, 1e-6)
    hint = ("use the ITU-R P.2040 / Sionna formulas given above, with the angle of incidence "
            "theta_i in radians, measured from the normal, and the complex eta.")
    if _close(rv, ref_h, 1e-6) and _close(rh, ref_v, 1e-6):
        hint = ("you swapped the polarizations: which field component lies in the plane of "
                "incidence for a vertical antenna above a horizontal ground?")
    elif _close(rv, -ref_v, 1e-6):
        hint = "check the sign of r_parallel: in the convention used here it tends to -1 at grazing incidence."
    _verdict("2.1", good, hint)


def check_ex2_2(two_ray_path_gain_db, distances, g_rt_db, h_t, h_r, frequency, eta):
    ok, g = _attempt("2.2", two_ray_path_gain_db, distances, h_t, h_r, frequency, eta, "V")
    if not ok:
        return
    g = np.asarray(g, dtype=float)
    lam = speed_of_light/frequency
    friis_db = 20*np.log10(lam/(4*np.pi*np.sqrt(distances**2 + (h_t - h_r)**2)))
    away = g_rt_db > friis_db - 10                       # away from the deep interference nulls
    if g.shape != g_rt_db.shape:
        _verdict("2.2", False, "return one path gain [dB] per distance (a NumPy array).")
        return
    diff = g[away] - g_rt_db[away]
    good = np.max(np.abs(diff)) < 0.5
    hint = ("check r1, r2, the angle of incidence (cos theta_i = (h_t + h_r) / r2), the phase "
            "factors exp(-j k r) and the reflection coefficient (Ex 2.1).")
    if np.std(diff) < 0.1 and abs(np.mean(diff)) > 0.5:
        hint = "your curve has the right shape but a constant offset: check the factor (lambda/4 pi)^2 and the dB conversion."
    _verdict("2.2", good, hint)


def check_ex2_3(breakpoint_distance, path_loss_exponent, distances, g_rt_db, h_t, h_r, frequency):
    ok, db1 = _attempt("2.3", breakpoint_distance, 3.0, 1.5, 3.5e9)
    if not ok:
        return
    ok, db2 = _attempt("2.3", breakpoint_distance, 10.0, 2.0, 1e9)
    if not ok:
        return
    good_db = abs(db1 - _REF["d_b"][0]) < 0.1 and abs(db2 - _REF["d_b"][1]) < 0.1
    d = np.logspace(1, 3, 61)
    g = np.where(d < 100, -20 - 20*np.log10(d), -60 - 40*np.log10(d/100))   # n = 2, then n = 4
    ok, n_a = _attempt("2.3", path_loss_exponent, d, g, 10.0, 100.0)
    if not ok:
        return
    ok, n_b = _attempt("2.3", path_loss_exponent, d, g, 100.0, 1000.0)
    if not ok:
        return
    good_fit = abs(n_a - 2.0) < 1e-6 and abs(n_b - 4.0) < 1e-6
    if not good_db:
        _verdict("2.3", False, "the breakpoint distance is d_b = 4 h_t h_r / lambda.")
        return
    if not good_fit:
        hint = "fit G_dB = A - 10 n log10(d) by least squares (np.polyfit) on the points with d_min <= d <= d_max."
        if abs(n_a - 3.0) < 1e-6 and abs(n_b - 3.0) < 1e-6:
            hint = "use only the points with d_min <= d <= d_max."
        elif abs(n_a + 20.0) < 1e-6:
            hint = "np.polyfit returns the slope in dB per decade: n = -slope / 10."
        _verdict("2.3", False, hint)
        return
    n1 = path_loss_exponent(distances, g_rt_db, 10.0, db1/2)
    n2 = path_loss_exponent(distances, g_rt_db, 2*db1, 1000.0)
    (lo1, hi1), (lo2, hi2) = _REF["n_range"]
    _verdict("2.3", lo1 < n1 < hi1 and lo2 < n2 < hi2,
             "your functions pass the tests, but the exponents of the ray-traced curve are off: "
             "did you modify the given data?")


def check_ex3_1(compute_paths):
    scene = make_canyon_scene()
    ok, p = _attempt("3.1", compute_paths, scene)
    if not ok:
        return
    if not hasattr(p, "cir"):
        _verdict("3.1", False, "compute_paths must return the Paths object returned by PathSolver.")
        return
    g = [10*np.log10(np.sum(np.abs(paths_of_rx(p, rx)[0])**2)) for rx in (0, 1)]
    ok, p_nd = _attempt("3.1", compute_paths, scene, diffraction=False)
    if not ok:
        return
    a_nd, _, types_nd = paths_of_rx(p_nd, 1)
    g_nd = 10*np.log10(np.sum(np.abs(a_nd)**2))
    diffraction_used = any("D" in t for t in paths_of_rx(p, 1)[2])
    diffraction_off_ok = not any("D" in t for t in types_nd)
    ref = _REF["g_31"]
    depth_ok = p.interactions.shape[0] == 5            # first dimension of interactions = max_depth
    good = (depth_ok and abs(g[0] - ref[0]) < 0.6 and abs(g[1] - ref[1]) < 0.6
            and abs(g_nd - ref[2]) < 0.6 and diffraction_used and diffraction_off_ok)
    hint = ("use PathSolver with max_depth=5, los=True, the specular_reflection and diffraction "
            "arguments of compute_paths, refraction=False, samples_per_src=10**6 and seed=SEED.")
    if not diffraction_off_ok or not diffraction_used:
        hint = "pass the diffraction argument of compute_paths on to PathSolver."
    if not depth_ok:
        hint = "use max_depth=5 (up to 5 interactions per path)."
    if g[1] > _REF["refraction_hint"]:
        hint = "the NLoS receiver gets too much power: is refraction (transmission through the walls) disabled?"
    _verdict("3.1", good, hint)


def check_ex3_2(rms_delay_spread):
    tau = np.array([10e-9, 60e-9, 260e-9])
    a = np.array([1.0, np.sqrt(0.1)*1j, -np.sqrt(0.01)])
    ok, t1 = _attempt("3.2", rms_delay_spread, np.array([0.0, 100e-9]), np.array([1.0, 1.0]))
    if not ok:
        return
    ok, t2 = _attempt("3.2", rms_delay_spread, tau, a)
    if not ok:
        return
    ok, t3 = _attempt("3.2", rms_delay_spread, tau + 1e-6, a)
    if not ok:
        return
    ok, t4 = _attempt("3.2", rms_delay_spread, tau, a, threshold_db=15.0)
    if not ok:
        return
    ok, t5 = _attempt("3.2", rms_delay_spread, np.array([0.0, 50e-9, 300e-9]),
                      np.array([1.0, 1.0j, 0.2]), threshold_db=14.0)
    if not ok:
        return
    good = (abs(t1 - 50e-9) < 1e-12 and abs(t2 - 27.25132e-9) < 1e-13 and abs(t3 - 27.25132e-9) < 1e-13
            and abs(t4 - 14.37399e-9) < 1e-13 and abs(t5 - 45.45891e-9) < 1e-13)
    hint = "tau_rms is the power-weighted standard deviation of the delays, with powers P_i = |a_i|^2."
    if abs(t2 - 64.35011e-9) < 1e-12:
        hint = "weight the delays with the powers |a_i|^2, not with the amplitudes |a_i|."
    elif abs(t2 - 31.99099e-9) < 1e-12:
        hint = "subtract the mean delay: tau_rms^2 = mean(tau^2) - mean(tau)^2."
    elif abs(t2 - 27.25132e-9) < 1e-13 and (abs(t4 - 14.37399e-9) > 1e-13 or abs(t5 - 45.45891e-9) > 1e-13):
        hint = ("with threshold_db, keep only the paths whose power is at most threshold_db below "
                "the strongest path: 10 log10(P_i / P_max) >= -threshold_db.")
    _verdict("3.2", good, hint)


def check_ex3_3(coherence_bandwidth_hz):
    ok, b1 = _attempt("3.3", coherence_bandwidth_hz, 25e-9)
    if not ok:
        return
    ok, b2 = _attempt("3.3", coherence_bandwidth_hz, 100e-9)
    if not ok:
        return
    good = abs(b1 - 8e6) < 1 and abs(b2 - 2e6) < 1
    hint = "use the rule of thumb for a frequency correlation of 0.5, in Hz."
    if abs(b1 - 0.8e6) < 1:
        hint = "1/(50 tau_rms) is the rule for a correlation of 0.9; here we use the 0.5 correlation."
    _verdict("3.3", good, hint)


def check_ex3_4(gains_by_configuration):
    ok, res = _attempt("3.4", gains_by_configuration, make_canyon_scene())
    if not ok:
        return
    ref = _REF["g_34"]
    if not isinstance(res, dict) or set(res) != set(ref):
        _verdict("3.4", False, f"return a dictionary with the keys {sorted(ref)}.")
        return
    good = all(abs(res[k] - ref[k]) < 0.6 for k in ref)
    _verdict("3.4", good, "call compute_paths with the right flags (diffraction=False, or "
                          "specular_reflection=False) and use the total path gain of the NLoS receiver.")


def check_ex4_1(compute_radio_map):
    scene = make_canyon_scene()
    ok, rm = _attempt("4.1", compute_radio_map, scene, 1)
    if not ok:
        return
    if not hasattr(rm, "path_gain"):
        _verdict("4.1", False, "return the radio map object computed by RadioMapSolver.")
        return
    g = rm.path_gain.numpy()
    if g.shape != (1, 122, 187):
        _verdict("4.1", False, "use cell_size=(1.0, 1.0) and let the map cover the whole scene "
                               "(do not pass center, orientation or size).")
        return
    reached = np.mean(g > 0)*100
    _verdict("4.1", abs(reached - _REF["reached_41"]) < 1.5,
             "use samples_per_tx=10**7, the max_depth, diffraction and refraction arguments, and seed=SEED.")


def check_ex4_2(watt_to_dbm):
    ok, v = _attempt("4.2", watt_to_dbm, np.array([1e-3, 1.0, 1e-12]))
    if not ok:
        return
    good = _close(v, np.array([0.0, 30.0, -90.0]), 1e-6)
    hint = "P[dBm] = 10 log10(P / 1 mW)."
    if _close(v, np.array([-30.0, 0.0, -120.0]), 1e-6):
        hint = "you returned dBW: dBm is relative to 1 mW."
    _verdict("4.2", good, hint)


def check_ex4_3(coverage_percentage):
    prx = np.array([[-80.0, -95.0, -90.0, -120.0], [-60.0, -91.0, -89.9, -np.inf],
                    [-85.0, -100.0, -70.0, -90.0], [-50.0, -50.0, -150.0, -88.0]])
    outdoor = np.array([[1, 1, 1, 0], [1, 1, 1, 1], [0, 1, 1, 1], [1, 0, 0, 1]], dtype=bool)
    ok, c = _attempt("4.3", coverage_percentage, prx, outdoor, -90.0)
    if not ok:
        return
    good = abs(c - 66.666667) < 1e-4
    hint = "percentage (0-100) of the outdoor cells with P_rx >= threshold."
    if abs(c - 0.6666667) < 1e-4:
        hint = "return a percentage (0-100), not a fraction."
    elif abs(c - 62.5) < 1e-4:
        hint = "count only the outdoor cells (use the boolean mask)."
    elif abs(c - 50.0) < 1e-4:
        hint = "a cell exactly at the threshold is covered (>=)."
    _verdict("4.3", good, hint)


def check_ex4_4(coverage_with_refraction):
    scene = make_canyon_scene()
    rm_function = globals().get("compute_radio_map")
    if rm_function is None:
        print("Exercise 4.1 not completed yet.")
        return
    ok, rm0 = _attempt("4.1", rm_function, scene, 0)
    if not ok:
        return
    ok, c = _attempt("4.4", coverage_with_refraction, scene, outdoor_mask(scene, rm0), -90.0)
    if not ok:
        return
    _verdict("4.4", abs(c - _REF["coverage_44"]) < 1.5, "compute the map with max_depth=5, diffraction off and "
                                         "refraction on, then the coverage at the threshold.")


def _check_ex5(bonus_function, reference):
    ok, value = _attempt("5", bonus_function)
    if not ok:
        return
    _verdict("5", abs(value - reference) < 1.5, "check the frequency or the materials of your scene, "
                                                "and use the same settings as in Ex 4 (max_depth=5, diffraction on).")

def check_ex5_a(bonus_coverage_28ghz):
    _check_ex5(bonus_coverage_28ghz, _REF["coverage_5a"])

def check_ex5_b(bonus_coverage_metal):
    _check_ex5(bonus_coverage_metal, _REF["coverage_5b"])

## Exercise 0 - Setup and environment check (5 min)

Run the check below. It prints the version and the variant (GPU or CPU) and computes one path.
On Colab, if a GPU runtime is not available, the CPU variant is used automatically: everything in
this lab also runs on a CPU, just more slowly.

In [ ]:
check_ex0()

## Exercise 1 - Free space: the Friis formula (15 min)

With isotropic antennas the free-space path gain at distance $d$ is
$$G_\mathrm{FS}=\left(\frac{\lambda}{4\pi d}\right)^2,\qquad
G_\mathrm{FS,dB}=20\log_{10}\frac{\lambda}{4\pi d},\qquad \lambda=\frac{c}{f}.$$
The ray tracer returns, for every path $i$, a complex coefficient $a_i$; the path gain of a
receiver is the sum of the path powers, $G=\sum_i\lvert a_i\rvert^2$.

The next cell (given) places a transmitter and 7 receivers at 10 m height in an **empty** scene, at
distances from 10 m to 1 km, and computes the line-of-sight paths at 3.5 GHz.

In [ ]:
F_EX1 = 3.5e9                                                       # carrier frequency [Hz]
D_EX1 = np.array([10.0, 20.0, 50.0, 100.0, 200.0, 500.0, 1000.0])   # distances [m]

scene1 = load_scene()                                               # empty scene: free space
scene1.frequency = F_EX1
scene1.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene1.rx_array = scene1.tx_array
scene1.add(Transmitter(name="tx", position=[0.0, 0.0, 10.0]))
for i, d in enumerate(D_EX1):
    scene1.add(Receiver(name=f"rx{i}", position=[float(d), 0.0, 10.0]))

paths1 = PathSolver()(scene1, max_depth=0, seed=SEED)
A_EX1 = paths1.cir(out_type="numpy")[0][:, 0, 0, 0, :, 0]          # [receiver, path] coefficients
print("coefficients a_i, one row per receiver:", A_EX1.shape)

**Before running the comparison: what do you expect?** By how many dB per decade of distance
should the path gain decrease, and what value do you expect at 1 km?

**Your answer:** _write here_

In [ ]:
def friis_path_gain_db(d_m, f_hz):
    '''Free-space path gain [dB] between isotropic antennas.
      d_m:  distance(s) [m], float or NumPy array
      f_hz: carrier frequency [Hz]
    Returns G_FS,dB (negative), same shape as d_m. Hint: speed_of_light is already imported.
    '''
    # TODO [Ex 1.1]: implement the Friis path gain in dB
    raise NotImplementedError("Ex 1.1 not completed yet: implement the Friis path gain in dB")


def path_gain_db(a):
    '''Path gain [dB] of one receiver: G = sum_i |a_i|^2, from the complex coefficients a (1D array).
    '''
    # TODO [Ex 1.2]: compute the path gain in dB from the path coefficients
    raise NotImplementedError("Ex 1.2 not completed yet: compute the path gain in dB from the path coefficients")

In [ ]:
check_ex1_1(friis_path_gain_db)
check_ex1_2(path_gain_db)

In [ ]:
g_rt_ex1 = np.array([path_gain_db(a) for a in A_EX1])
g_fs_ex1 = friis_path_gain_db(D_EX1, F_EX1)
print(f"{'d [m]':>7s} {'RT [dB]':>9s} {'Friis [dB]':>11s} {'RT - Friis [dB]':>16s}")
for d, g1, g2 in zip(D_EX1, g_rt_ex1, g_fs_ex1):
    print(f"{d:7.0f} {g1:9.3f} {g2:11.3f} {g1 - g2:16.2e}")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.semilogx(D_EX1, g_fs_ex1, "k-", label="Friis")
ax.semilogx(D_EX1, g_rt_ex1, "o", mfc="none", label="Sionna RT")
ax.set(xlabel="distance [m]", ylabel="path gain [dB]", title="Free space, 3.5 GHz")
ax.legend();

In [ ]:
check_ex1_3(friis_path_gain_db, path_gain_db, A_EX1, D_EX1, F_EX1)

**Q1.1** What slope do you observe, in dB per decade, and which path-loss exponent does it
correspond to? Why do the ray tracer and Friis agree so closely?

**Your answer:** _write here_

**Q1.2** At 28 GHz, at which distance is the path gain equal to the path gain at 3.5 GHz and
100 m? Compute it (you can use your function) and explain the result with the Friis formula.

**Your answer:** _write here_

## Exercise 2 - Two-ray model over a real ground (30 min)

A transmitter at height $h_t$ and a receiver at height $h_r$, at horizontal distance $d$ over a
flat ground, receive the direct ray (length $r_1$) and the ray reflected by the ground (length
$r_2$, angle of incidence $\theta_i$ measured **from the normal** of the ground):
$$G=\left(\frac{\lambda}{4\pi}\right)^2\left|\frac{e^{-jkr_1}}{r_1}+R(\theta_i)\,
\frac{e^{-jkr_2}}{r_2}\right|^2,\qquad r_{1,2}=\sqrt{d^2+(h_t\mp h_r)^2},\qquad
\cos\theta_i=\frac{h_t+h_r}{r_2},\qquad k=\frac{2\pi}{\lambda}.$$
Sionna uses the ITU-R P.2040 convention for the Fresnel coefficients of a ground with complex
relative permittivity $\eta=\varepsilon_r-j\sigma/(\varepsilon_0\omega)$:
$$r_\perp=\frac{\cos\theta_i-\sqrt{\eta-\sin^2\theta_i}}{\cos\theta_i+\sqrt{\eta-\sin^2\theta_i}}
\ \ (\text{TE: } \mathbf E\perp\text{ plane of incidence}),\qquad
r_\parallel=\frac{\eta\cos\theta_i-\sqrt{\eta-\sin^2\theta_i}}{\eta\cos\theta_i+\sqrt{\eta-\sin^2\theta_i}}
\ \ (\text{TM: } \mathbf E\text{ in the plane of incidence}).$$
The **breakpoint distance** is $d_b=4h_th_r/\lambda$.

The given cell builds a flat ground (ITU *medium dry ground*, a 1 m thick slab, so that it reflects
like a half-space), with **vertically polarized** isotropic antennas at $h_t=3$ m and $h_r=1.5$ m,
and computes the ray-traced path gain at 3.5 GHz for 300 distances from 10 m to 1 km. The two rays
are added coherently, as they interfere at a single frequency.

In [ ]:
F_EX2, H_T, H_R = 3.5e9, 3.0, 1.5                 # frequency [Hz], antenna heights [m]
D_EX2 = np.logspace(1, 3, 300)                    # horizontal distances [m]

ground_mat = ITURadioMaterial("ground-ex2", "medium_dry_ground", thickness=1.0)
scene2 = load_scene()
scene2.edit(add=[SceneObject(mi_mesh=flat_ground(-50.0, 1050.0, -100.0, 100.0), name="ground",
                             radio_material=ground_mat)])
scene2.frequency = F_EX2
scene2.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene2.rx_array = scene2.tx_array
scene2.add(Transmitter(name="tx", position=[0.0, 0.0, H_T]))
for i, d in enumerate(D_EX2):
    scene2.add(Receiver(name=f"rx{i}", position=[float(d), 0.0, H_R]))

paths2 = PathSolver()(scene2, max_depth=1, los=True, specular_reflection=True, refraction=False,
                      seed=SEED)
a2 = paths2.cir(out_type="numpy")[0][:, 0, 0, 0, :, 0]
G_RT_EX2_DB = 10*np.log10(np.abs(a2.sum(axis=-1))**2)   # coherent sum: LoS + ground reflection
ETA_GROUND = (ground_mat.relative_permittivity[0]
              - 1j*ground_mat.conductivity[0]/(epsilon_0*2*np.pi*F_EX2))
print(f"complex relative permittivity of the ground at 3.5 GHz: eta = {ETA_GROUND:.3f}")

**Before plotting: what do you expect?** How should the path gain behave below and above the
breakpoint distance? Where do you expect deep minima?

**Your answer:** _write here_

In [ ]:
def reflection_coefficient(theta_i, eta, polarization):
    '''Fresnel reflection coefficient of the ground (ITU-R P.2040 / Sionna convention).
      theta_i: angle(s) of incidence [rad] from the normal; eta: complex relative permittivity;
      polarization: "V" or "H", the polarization of the antennas (vertical or horizontal).
    Returns the complex coefficient: r_parallel or r_perpendicular, whichever applies.
    '''
    # TODO [Ex 2.1]: return the Fresnel coefficient that applies to the polarization
    raise NotImplementedError("Ex 2.1 not completed yet: return the Fresnel coefficient that applies to the polarization")

In [ ]:
check_ex2_1(reflection_coefficient)

In [ ]:
def two_ray_path_gain_db(d, h_t, h_r, f_hz, eta, polarization):
    '''Two-ray path gain [dB] (coherent sum of direct and ground-reflected rays) for horizontal
    distances d [m] (NumPy array), antenna heights h_t, h_r [m], frequency f_hz [Hz],
    ground permittivity eta and antenna polarization ("V" or "H"). Use reflection_coefficient.
    '''
    # TODO [Ex 2.2]: implement the two-ray path gain in dB
    raise NotImplementedError("Ex 2.2 not completed yet: implement the two-ray path gain in dB")

In [ ]:
check_ex2_2(two_ray_path_gain_db, D_EX2, G_RT_EX2_DB, H_T, H_R, F_EX2, ETA_GROUND)

In [ ]:
g_v = two_ray_path_gain_db(D_EX2, H_T, H_R, F_EX2, ETA_GROUND, "V")
g_h = two_ray_path_gain_db(D_EX2, H_T, H_R, F_EX2, ETA_GROUND, "H")
g_fs = 20*np.log10(speed_of_light/F_EX2/(4*np.pi*np.sqrt(D_EX2**2 + (H_T - H_R)**2)))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogx(D_EX2, g_fs, "k:", label="Friis")
ax.semilogx(D_EX2, G_RT_EX2_DB, "-", lw=3, alpha=0.5, label="Sionna RT (V antennas)")
ax.semilogx(D_EX2, g_v, "--", color="C3", label="your two-ray model, V")
ax.semilogx(D_EX2, g_h, "-", lw=0.8, color="C2", label="your two-ray model, H (for comparison)")
ax.set(xlabel="horizontal distance d [m]", ylabel="path gain [dB]", ylim=(-130, -50),
       title="Two-ray model, h_t = 3 m, h_r = 1.5 m, 3.5 GHz")
ax.legend(loc="lower left");

In [ ]:
def breakpoint_distance(h_t, h_r, f_hz):
    '''Breakpoint distance d_b [m] of the two-ray model.
    '''
    # TODO [Ex 2.3]: return the breakpoint distance d_b
    raise NotImplementedError("Ex 2.3 not completed yet: return the breakpoint distance d_b")


def path_loss_exponent(d, g_db, d_min, d_max):
    '''Least-squares fit of g_db = A - 10 n log10(d) over the points with d_min <= d <= d_max.
    Returns the path-loss exponent n. Hint: np.polyfit(x, y, 1) returns (slope, intercept).
    '''
    # TODO [Ex 2.3]: fit the path-loss exponent n over [d_min, d_max]
    raise NotImplementedError("Ex 2.3 not completed yet: fit the path-loss exponent n over [d_min, d_max]")

In [ ]:
check_ex2_3(breakpoint_distance, path_loss_exponent, D_EX2, G_RT_EX2_DB, H_T, H_R, F_EX2)

In [ ]:
d_b = breakpoint_distance(H_T, H_R, F_EX2)
n1 = path_loss_exponent(D_EX2, G_RT_EX2_DB, 10.0, d_b/2)
n2 = path_loss_exponent(D_EX2, G_RT_EX2_DB, 2*d_b, 1000.0)
print(f"breakpoint distance d_b = {d_b:.1f} m")
print(f"path-loss exponent from the ray-traced curve: n1 = {n1:.2f} for 10 m <= d <= d_b/2, "
      f"n2 = {n2:.2f} for 2 d_b <= d <= 1 km")
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(D_EX2, G_RT_EX2_DB, "-", lw=2, alpha=0.6, label="Sionna RT")
for lo, hi, n, color in [(10.0, d_b/2, n1, "C1"), (2*d_b, 1000.0, n2, "C3")]:
    sel = (D_EX2 >= lo) & (D_EX2 <= hi)
    a_fit = np.mean(G_RT_EX2_DB[sel] + 10*n*np.log10(D_EX2[sel]))
    ax.semilogx(D_EX2[sel], a_fit - 10*n*np.log10(D_EX2[sel]), color=color, lw=2, label=f"fit, n = {n:.2f}")
ax.axvline(d_b, color="gray", ls="--")
ax.set(xlabel="horizontal distance d [m]", ylabel="path gain [dB]", ylim=(-130, -50),
       title="Path-loss exponents before and after the breakpoint")
ax.legend(loc="lower left");

**Q2.1** Why is the TM coefficient $r_\parallel$ the right one for the vertically polarized
antennas of this scene?

**Your answer:** _write here_

**Q2.2** Around which distance do the interference ripples almost disappear for the V model? Compute
the grazing angle $\psi=\arctan\big((h_t+h_r)/d\big)$ there and relate it to the ground. Does this
happen with H polarization? Why?

**Your answer:** _write here_

**Q2.3** Why does the exponent tend to 4 beyond the breakpoint, and why is your fitted $n_2$
slightly smaller than 4?

**Your answer:** _write here_

## Exercise 3 - Multipath in a street canyon (35 min)

The scene `simple_street_canyon` has six buildings (glass, wood, marble and brick, walls 0.1 m
thick) on a concrete floor. A small cell on a 10 m pole in the main street transmits 30 dBm at
3.5 GHz. Receiver **RX-LoS** is in the main street, 70 m away; receiver **RX-NLoS** is in a cross
street, hidden by the corner of a building. Transmission through the walls (refraction) is
disabled: Exercise 4.4 shows why.

From now on **you also write the Sionna calls**. Useful pages:
[PathSolver](https://nvlabs.github.io/sionna/rt/api/paths_solvers.html),
[Paths](https://nvlabs.github.io/sionna/rt/api/paths.html).

In [ ]:
scene3 = make_canyon_scene()
print("transmitter:", TX_CANYON, "| RX-LoS:", RX_LOS, "| RX-NLoS:", RX_NLOS)
scene3.render(camera=cam_canyon, resolution=(640, 480), num_samples=32);

**Before running: what do you expect?** Which receiver will have the larger RMS delay spread? At
RX-NLoS, which mechanism do you expect to bring most of the power: reflections or diffraction?

**Your answer:** _write here_

In [ ]:
def compute_paths(scene, diffraction=True, specular_reflection=True):
    '''Propagation paths of `scene` with PathSolver: line of sight on, up to 5 interactions,
    specular reflection and diffraction as given by the arguments, refraction off,
    10**6 rays per source (samples_per_src) and seed=SEED. Returns the Paths object.
    '''
    # TODO [Ex 3.1]: call PathSolver with the requested settings and return the paths
    raise NotImplementedError("Ex 3.1 not completed yet: call PathSolver with the requested settings and return the paths")

In [ ]:
check_ex3_1(compute_paths)

In [ ]:
def rms_delay_spread(tau_s, a, threshold_db=None):
    '''RMS delay spread [s] of the paths with delays tau_s [s] and complex coefficients a.
    If threshold_db is given, keep only the paths whose power is at most threshold_db below
    the strongest path. Remember: the power of a path is |a_i|^2.
    '''
    # TODO [Ex 3.2]: compute the RMS delay spread, with the optional threshold
    raise NotImplementedError("Ex 3.2 not completed yet: compute the RMS delay spread, with the optional threshold")

In [ ]:
check_ex3_2(rms_delay_spread)

In [ ]:
paths3 = compute_paths(scene3)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, rx, name in zip(axes, [0, 1], ["RX-LoS", "RX-NLoS"]):
    a, tau, types = paths_of_rx(paths3, rx)
    p_db = 10*np.log10(np.abs(a)**2)
    tau_ns = (tau - tau.min())*1e9
    for label, select, color in [("LoS", lambda t: t == "LoS", "C3"),
                                 ("reflections only", lambda t: t != "LoS" and "D" not in t, "C0"),
                                 ("with a diffraction", lambda t: "D" in t, "C2")]:
        sel = np.array([select(t) for t in types])
        if sel.any():
            ax.vlines(tau_ns[sel], p_db.max() - 50, p_db[sel], color=color, lw=1)
            ax.plot(tau_ns[sel], p_db[sel], "o", ms=3, color=color, label=label)
    t_all, t_20 = rms_delay_spread(tau, a), rms_delay_spread(tau, a, threshold_db=20.0)
    print(f"{name}: {len(a)} paths, G = {10*np.log10(np.sum(np.abs(a)**2)):.1f} dB, "
          f"tau_rms = {t_all*1e9:.1f} ns (paths within 20 dB of the strongest: {t_20*1e9:.1f} ns)")
    ax.set(xlabel="excess delay [ns]", ylabel="path power [dB]", ylim=(p_db.max() - 50, p_db.max() + 3),
           title=f"{name}: power delay profile")
    ax.legend(fontsize=8)

In [ ]:
def coherence_bandwidth_hz(tau_rms_s):
    '''Coherence bandwidth [Hz] for a frequency correlation of 0.5, from the rule of thumb
    based on the RMS delay spread tau_rms_s [s].
    '''
    # TODO [Ex 3.3]: return the rule-of-thumb coherence bandwidth
    raise NotImplementedError("Ex 3.3 not completed yet: return the rule-of-thumb coherence bandwidth")

In [ ]:
check_ex3_3(coherence_bandwidth_hz)

In [ ]:
paths3 = compute_paths(scene3)
freqs = subcarrier_frequencies(1024, 100e6/1024)          # 100 MHz around the carrier
h_f = paths3.cfr(frequencies=freqs, normalize_delays=True, out_type="numpy")[:, 0, 0, 0, 0, :]
fig, ax = plt.subplots(figsize=(9, 4))
for rx, name, color in [(0, "RX-LoS", "C0"), (1, "RX-NLoS", "C1")]:
    a, tau, _ = paths_of_rx(paths3, rx)
    b_c = coherence_bandwidth_hz(rms_delay_spread(tau, a))
    print(f"{name}: coherence bandwidth B_c = {b_c/1e6:.1f} MHz")
    ax.plot(freqs.numpy()/1e6, 20*np.log10(np.abs(h_f[rx])), color=color,
            label=f"{name} (B_c = {b_c/1e6:.1f} MHz)")
    ax.axvspan(-b_c/2e6, b_c/2e6, color=color, alpha=0.12)
ax.set(xlabel="frequency offset from 3.5 GHz [MHz]", ylabel="$|H(f)|^2$ [dB]",
       title="Channel frequency response over 100 MHz (shaded: width B_c)")
ax.legend();

In [ ]:
def gains_by_configuration(scene):
    '''Total path gain [dB] of RX-NLoS (receiver index 1) in three configurations, as a dictionary
    {"all": ..., "no diffraction": ..., "no reflection": ...}. Use compute_paths and paths_of_rx;
    "no reflection" means specular_reflection=False (diffraction stays on).
    '''
    # TODO [Ex 3.4]: compute the NLoS path gain with all mechanisms and with one disabled
    raise NotImplementedError("Ex 3.4 not completed yet: compute the NLoS path gain with all mechanisms and with one disabled")

In [ ]:
check_ex3_4(gains_by_configuration)

In [ ]:
for label, g in gains_by_configuration(scene3).items():
    print(f"RX-NLoS, {label:15s}: G = {g:7.1f} dB")

**Q3.1** Which mechanism dominates at RX-NLoS? Explain why, using the geometry of the street
canyon (look at the rendered scene and at the positions).

**Your answer:** _write here_

**Q3.2** Compare your RMS delay spreads with the 3GPP TR 38.901 medians for the urban micro street
canyon scenario at 3.5 GHz: about 40 ns (LoS) and 97 ns (NLoS). Are they of the same order? Why
are yours smaller?

**Your answer:** _write here_

**Q3.3** A 5G NR carrier at 3.5 GHz typically has 100 MHz of bandwidth and 30 kHz subcarriers.
Compare your coherence bandwidths with these two numbers: is the channel flat or
frequency-selective over the carrier? Over one subcarrier? Does the plot of $|H(f)|$ agree?

**Your answer:** _write here_

**Q3.4** How much does $\tau_\mathrm{rms}$ change if you keep only the paths within 20 dB of the
strongest one? What does this tell you about measured delay spreads?

**Your answer:** _write here_

## Exercise 4 - Radio map and coverage (30 min)

A radio map gives, for every cell of a horizontal plane at 1.5 m, the cell-averaged path gain
$g$ (non-coherent sum of the path powers, isotropic receiver) and the received power
`rm.rss` $=P_\mathrm{tx}\,g$ **in watts**, with shape `[num_tx, num_cells_y, num_cells_x]`.
We define the **coverage** as the percentage of *outdoor* cells where
$P_\mathrm{rx}\ge -90$ dBm (transmit power 30 dBm). Useful page:
[RadioMapSolver](https://nvlabs.github.io/sionna/rt/api/radio_map_solvers.html).

In [ ]:
scene4 = make_canyon_scene()
THRESHOLD_DBM = -90.0
print(f"transmit power {P_TX_DBM:.0f} dBm, coverage threshold {THRESHOLD_DBM:.0f} dBm")

**Before running: what do you expect?** How should the coverage change when the maximum number of
interactions grows from 0 to 5? And when diffraction is added?

**Your answer:** _write here_

In [ ]:
def compute_radio_map(scene, max_depth, diffraction=False, refraction=False):
    '''Radio map of `scene` with RadioMapSolver: the default horizontal plane covering the whole
    scene at 1.5 m (do not pass center, orientation or size), 1 m x 1 m cells (cell_size),
    10**7 rays (samples_per_tx), the given max_depth, diffraction and refraction, seed=SEED.
    '''
    # TODO [Ex 4.1]: call RadioMapSolver with the requested settings and return the map
    raise NotImplementedError("Ex 4.1 not completed yet: call RadioMapSolver with the requested settings and return the map")

In [ ]:
check_ex4_1(compute_radio_map)

In [ ]:
def watt_to_dbm(p_w):
    '''Convert powers p_w [W] (NumPy array) to dBm. Cells without signal (0 W) become -inf.
    '''
    # TODO [Ex 4.2]: convert watts to dBm
    raise NotImplementedError("Ex 4.2 not completed yet: convert watts to dBm")


def coverage_percentage(prx_dbm, outdoor, threshold_dbm):
    '''Percentage (0-100) of the outdoor cells (boolean mask `outdoor`) where prx_dbm >= threshold_dbm.
    '''
    # TODO [Ex 4.3]: compute the coverage percentage over the outdoor cells
    raise NotImplementedError("Ex 4.3 not completed yet: compute the coverage percentage over the outdoor cells")

In [ ]:
check_ex4_2(watt_to_dbm)
check_ex4_3(coverage_percentage)

In [ ]:
configs = [("LoS only", 0, False), ("max_depth=1", 1, False), ("max_depth=3", 3, False),
           ("max_depth=5", 5, False), ("max_depth=5 + diffraction", 5, True)]
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
coverage, reached = [], []
for ax, (label, depth, diffraction) in zip(axes.ravel(), configs):
    rm = compute_radio_map(scene4, depth, diffraction=diffraction)
    if depth == 0:
        outdoor4 = outdoor_mask(scene4, rm)
    prx = watt_to_dbm(rm.rss.numpy()[0])
    coverage.append(coverage_percentage(prx, outdoor4, THRESHOLD_DBM))
    reached.append(np.mean(np.isfinite(prx[outdoor4]))*100)   # outdoor cells with any power
    show_map(ax, prx, rm, outdoor4, f"{label}: coverage {coverage[-1]:.1f} %")
ax = axes.ravel()[-1]
ax.plot([0, 1, 3, 5], coverage[:4], "o-", label="specular reflections")
ax.plot([5], coverage[4:], "s", ms=8, label="+ diffraction")
ax.set(xlabel="max_depth", ylabel="coverage [%]", ylim=(0, 100),
       title=f"Outdoor coverage, P_rx >= {THRESHOLD_DBM:.0f} dBm")
ax.legend()
fig.tight_layout()
for (label, _, _), c, r in zip(configs, coverage, reached):
    print(f"{label:27s}: coverage {c:5.1f} %, outdoor cells reached by at least one ray {r:5.1f} %")

In [ ]:
def coverage_with_refraction(scene, outdoor, threshold_dbm):
    '''Coverage [%] with max_depth=5, diffraction off and refraction ON (transmission through
    the walls). Use your functions compute_radio_map, watt_to_dbm and coverage_percentage.
    '''
    # TODO [Ex 4.4]: compute the coverage with refraction enabled
    raise NotImplementedError("Ex 4.4 not completed yet: compute the coverage with refraction enabled")

In [ ]:
check_ex4_4(coverage_with_refraction)

In [ ]:
rm_off = compute_radio_map(scene4, 5, refraction=False)
rm_on = compute_radio_map(scene4, 5, refraction=True)
outdoor4 = outdoor_mask(scene4, rm_off)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, rm, label in [(axes[0], rm_off, "refraction off"), (axes[1], rm_on, "refraction on")]:
    prx = watt_to_dbm(rm.rss.numpy()[0])
    indoor_reached = np.mean(rm.rss.numpy()[0][~outdoor4] > 0)*100
    show_map(ax, prx, rm, outdoor4, f"max_depth=5, {label}: coverage "
             f"{coverage_percentage(prx, outdoor4, THRESHOLD_DBM):.1f} %, "
             f"indoor cells reached {indoor_reached:.0f} %")
fig.tight_layout()

**Q4.1** Why is the LoS-only coverage (max_depth = 0) equal to the fraction of the outdoor area in
line of sight of the transmitter?

**Your answer:** _write here_

**Q4.2** Why does the coverage grow with max_depth, and why does going from 3 to 5 interactions add
less than going from 1 to 3?

**Your answer:** _write here_

**Q4.3** Diffraction reaches many new cells, but how much coverage does it add at $-90$ dBm? Why?

**Your answer:** _write here_

**Q4.4** With refraction enabled, how do the coverage and the map change? Why is this result
unrealistic for the buildings of this scene?

**Your answer:** _write here_

## Exercise 5 - Bonus: frequency or materials

Choose **one** option and write all the code yourself (you can reuse your functions from
Exercise 4). Use max_depth = 5 with diffraction, as in the last map of Ex 4.

- **Option A:** compute the coverage at **28 GHz** (`make_canyon_scene(frequency=28e9)`).
- **Option B:** make **all buildings metal** (keep the concrete floor) and compute the coverage at
  3.5 GHz. Hint: loop over `scene.objects.values()` and assign
  `ITURadioMaterial("buildings-metal", "metal", thickness=0.1)` to every object whose
  `radio_material.itu_type` is not `"concrete"`.

Then explain the change with respect to your result for the last map of Exercise 4.

In [ ]:
def bonus_coverage_28ghz():
    '''Option A: coverage [%] at 28 GHz, max_depth=5 with diffraction, threshold -90 dBm.
    '''
    # TODO [Ex 5]: compute the coverage at 28 GHz
    raise NotImplementedError("Ex 5 not completed yet: compute the coverage at 28 GHz")


def bonus_coverage_metal():
    '''Option B: coverage [%] at 3.5 GHz with all buildings made of metal, max_depth=5 with
    diffraction, threshold -90 dBm.
    '''
    # TODO [Ex 5]: compute the coverage with metal buildings
    raise NotImplementedError("Ex 5 not completed yet: compute the coverage with metal buildings")

In [ ]:
check_ex5_a(bonus_coverage_28ghz)    # option A
check_ex5_b(bonus_coverage_metal)    # option B

**Q5** Report your coverage and explain the change with respect to Exercise 4.

**Your answer:** _write here_

## How to submit

1. Make sure every part you completed prints **Correct** and every question has an answer.
2. Save the notebook with the outputs:
   - On Colab: *File → Download → Download .ipynb*.
   - On the lab computers: *File → Save* (Ctrl+S); the `.ipynb` file is in your working folder.
3. Rename it `lab_<surname1>[_<surname2>].ipynb` and upload it where your instructor tells you.